Multimodal AI models can understand and generate content across different types of data—text, images, video, audio, and documents—through a unified interface. This notebook explores these capabilities using Google's Gemini API as a practical example.We demonstrate core capabilities like text understanding, visual reasoning, and cross-modal tasks. The examples progress from fundamental operations to more advanced applications like object detection with bounding boxes, image segmentation, and solving mathematical problems from images.Each example is designed to be immediately runnable and applicable to real-world use cases.

# Setup and Configuration

First, install the required packages and set up your Gemini API key.

## Get Your API Key

1. Visit [Google AI Studio](https://aistudio.google.com/app/apikey)
2. Create or select a project
3. Generate an API key
4. Set it as an environment variable:

```bash
export GEMINI_API_KEY='your-api-key-here'
```


In [ ]:
# Install required packages
# Uncomment the line below to install dependencies
# !pip install -q google-genai pillow requests matplotlib pandas numpy


In [ ]:
import os
import json
import time
from pathlib import Path
from typing import List, Dict, Any
from google import genai
from PIL import Image, ImageDraw, ImageFont
import requests
from io import BytesIO
import base64
import matplotlib.pyplot as plt
import numpy as np

# Check for API key
if 'GEMINI_API_KEY' not in os.environ:
    raise ValueError(
        "GEMINI_API_KEY not found in environment.\n"
        "Set it with: export GEMINI_API_KEY='your-key'\n"
        "Get your key at: https://aistudio.google.com/apikey"
    )

# Initialize client (new SDK)
client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

print(" Gemini client initialized successfully")
print("Using google-genai SDK (new version)")

# Note: We'll use gemini-3-pro as the default model
MODEL = "models/gemini-3-pro-preview"
print(f"Default model: {MODEL}")

IMAGE_MODEL = "models/gemini-3-pro-image-preview"
print(f"Image model: {IMAGE_MODEL}")

%config InlineBackend.figure_format = 'retina'  # High-res plots

In [ ]:
# List all available models
print("Available Gemini models:")
print("="*80)
for model in client.models.list():
    if 'gemini' in model.name.lower():
        print(f"  - {model.name}")
print("\n" + "="*80)


**Note**: We're using `models/gemini-3-pro-preview` - the latest Gemini 3 Pro model available via the API. This model delivers the state-of-the-art multimodal capabilities described in Google's announcement.


## Helper Functions

In [ ]:
def print_section(title: str):
    """Print formatted section header."""
    print("\n" + "="*80)
    print(title)
    print("="*80)

def print_result(label: str, content: str, indent: int = 0):
    """Print formatted result."""
    prefix = "  " * indent
    print(f"{prefix}{label}: {content}")

def load_image_from_url(url: str) -> Image.Image:
    """Load an image from a URL."""
    response = requests.get(url)
    return Image.open(BytesIO(response.content))

def create_sample_image(text: str, size=(800, 600)) -> Image.Image:
    """Create a simple image with text for testing."""
    img = Image.new('RGB', size, color='white')
    draw = ImageDraw.Draw(img)
    draw.text((50, size[1]//2), text, fill='black')
    return img

print("Helper functions loaded")

## Single Request vs Batch Processing

In [ ]:
print("Single Request")
print("="*80)

# Single request - simplest way
response = client.models.generate_content(
    model=MODEL,
    contents="What is the capital of France?")
print_result("Question", "What is the capital of France?")
print_result("Answer", response.text)

In [ ]:
print("\n" + "="*80)
print("Sequential processing:")
print("="*80)

# Process multiple prompts efficiently
prompts = [
    "Translate 'Hello' to Spanish",
    "Translate 'Goodbye' to French",
]

start = time.time()
results_seq = []
for prompt in prompts:
    response = client.models.generate_content(
        model=MODEL,
        contents=prompt
    )
    results_seq.append(response.text.strip())
time_seq = time.time() - start

for i, (prompt, result) in enumerate(zip(prompts, results_seq), 1):
    print(f"  {i}. {prompt} → {result}")
print(f"Time: {time_seq:.2f}s")

# Text Understanding and NLP Tasks 

While Gemini-3-Pro excels at multimodal tasks, it also delivers exceptional performance on pure text tasks. These examples demonstrate fundamental NLP capabilities:


## Sentiment Analysis


In [ ]:
print("\n" + "="*80)
print("Zero-Shot Sentiment Analysis")
print("="*80)

texts = [
    "This product is absolutely amazing! Best purchase I've made all year.",
    "Terrible experience. Waste of money and time.",
    "It's okay. Nothing special but does the job.",
    "I'm disappointed with the quality. Expected much better.",
    "Exceeded all my expectations! Highly recommend!"
]

prompt_template = """Classify the sentiment: Positive, Negative, or Neutral.
Reply with ONLY the sentiment label.

Text: {text}
Sentiment:"""

for i, text in enumerate(texts, 1):
    response = client.models.generate_content(
        model=MODEL,
        contents=prompt_template.format(text=text)
    )
    sentiment = response.text.strip()
    print(f"{i}. '{text[:50]}...'")
    print(f"   → {sentiment}\n")

## Few-Shot Classification


In [ ]:
print("\n" + "="*80)
print("Few-Shot Text Classification")
print("="*80)

# Intent classification with examples
prompt = """Classify customer service queries into categories.

Examples:
"How do I reset my password?" → Technical Support
"I was charged twice" → Billing
"What are your hours?" → General Inquiry
"This is broken" → Complaint
"I want to cancel" → Account Management

Query: "{query}"
Category:"""

test_queries = [
    "My app keeps crashing when I upload photos",
    "Why was I charged for premium when I'm on free plan?",
    "Do you ship to Canada?",
    "The product arrived damaged",
    "How do I delete my account?"
]

for query in test_queries:
    response = client.models.generate_content(
        model=MODEL,
        contents=prompt.format(query=query)
    )
    print(f"Query: {query}")
    print(f"Category: {response.text.strip()}\n")


## Named Entity Recognition


In [ ]:
print("\n" + "="*80)
print("Named Entity Recognition (NER)")
print("="*80)

text = """Apple Inc. CEO Tim Cook announced a $500 million investment in renewable 
energy projects across California next month. The announcement was made at the 
company's headquarters in Cupertino on December 15, 2024."""

prompt = f"""Extract all named entities and categorize them:
PERSON, ORGANIZATION, LOCATION, MONEY, DATE

Text: {text}

Format as JSON with entity type as key."""

response = client.models.generate_content(
    model=MODEL,
    contents=prompt)
print(f"Text: {text}\n")
print("Entities:")
print(response.text)

## Text Summarization

In [ ]:
print("\n" + "="*80)
print("Text Summarization")
print("="*80)

article = """Artificial intelligence continues to transform industries worldwide. Recent
advances in large language models have enabled more natural conversations between humans
and machines. These models can understand context, generate coherent text, and even
perform complex reasoning tasks. However, challenges remain in ensuring factual accuracy,
reducing computational costs, and addressing ethical concerns around bias and privacy.
Researchers are actively working on making AI more efficient, transparent, and aligned
with human values. The field is evolving rapidly, with new breakthroughs announced weekly.
From healthcare to education, AI is reshaping how we work and live."""

prompts = [
    "Summarize in 1 sentence:",
    "Summarize in 3 bullet points:",
    "Create a tweet-length summary (280 chars):"
]

print(f"Original ({len(article)} chars):\n{article}\n")

for prompt_type in prompts:
    response = client.models.generate_content(
        model=MODEL,
        contents=f"{prompt_type}\n\n{article}"
    )
    print(f"{prompt_type}")
    print(f"  {response.text.strip()}\n")

## Question Answering


In [ ]:
print("\n" + "="*80)
print("Question Answering")
print("="*80)

context = """The Eiffel Tower is a wrought-iron lattice tower located on the Champ de Mars
in Paris, France. It was constructed from 1887 to 1889 as the centerpiece of the 1889
World's Fair. The tower is 330 meters (1,083 feet) tall, about the same height as an
81-story building. It was the tallest man-made structure in the world until the Chrysler
Building was completed in New York in 1930."""

questions = [
    "How tall is the Eiffel Tower?",
    "What material is it made of?",
    "When did it stop being the tallest structure?"
]

print(f"Context: {context}\n")

for q in questions:
    prompt = f"Context: {context}\n\nQuestion: {q}\nAnswer (concise):"
    response = client.models.generate_content(
        model=MODEL,
        contents=prompt
    )
    print(f"Q: {q}")
    print(f"A: {response.text.strip()}\n")

## Multi-Language Translation


In [ ]:
print("\n" + "="*80)
print("Multi-Language Translation")
print("="*80)

text = "Artificial intelligence is changing the world."
languages = ["Spanish", "Hindi"]

print(f"Original (English): {text}\n")

for lang in languages:
    prompt = f"Translate to {lang}: {text}"
    response = client.models.generate_content(
        model=MODEL,
        contents=prompt
    )
    print(f"{lang}: {response.text.strip()}")

## Object Detection with Bounding Boxes


In [ ]:
print("\n" + "="*80)
print("Object Detection with Bounding Boxes")
print("="*80)

import supervision as sv
import numpy as np
from PIL import Image
import requests
from io import BytesIO

# Load a cat image from Unsplash
cat_image_url = "https://images.unsplash.com/photo-1574158622682-e40e69881006?w=800"
print(f"Loading cat image from Unsplash...")

response = requests.get(cat_image_url)
image = Image.open(BytesIO(response.content))

print(f"Image loaded: {image.size[0]}x{image.size[1]} pixels")

# Display original image
plt.figure(figsize=(5, 4))
plt.imshow(image)
plt.title('Original Cat Image')
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Prompt Gemini to detect cat features
detection_prompt = """Detect the following objects in this image and return their bounding boxes:
1. The cat's left eye
2. The cat's right eye
3. The cat's left ear
4. The cat's right ear
5. The entire cat body

Return a JSON array where each object has:
- "label": object name (e.g., "left eye", "right eye", "left ear", "right ear", "cat")
- "box_2d": bounding box as [y_min, x_min, y_max, x_max] in range [0, 1000]

Format: [{"label": "left eye", "box_2d": [y0, x0, y1, x1]}, ...]

Return ONLY the JSON array, no other text."""

print("Asking Gemini to detect cat features (eyes, ears, body)...")
print()

response = client.models.generate_content(
    model=MODEL,
    contents=[detection_prompt, image]
)

print("Gemini Response:")
print(response.text)
print()

In [ ]:
# Parse the JSON response
import json
import re

response_text = response.text.strip()

# Remove markdown code blocks if present
if response_text.startswith('```'):
    lines = response_text.split('\n')
    response_text = '\n'.join(lines[1:-1])
    if response_text.startswith('json'):
        response_text = response_text[4:].strip()

try:
    detections_data = json.loads(response_text)
    
    print(f"Detected {len(detections_data)} objects:")
    for i, det in enumerate(detections_data, 1):
        print(f"  {i}. {det['label']}: box at {det['box_2d']}")
    print()
    
    # Convert Gemini's bounding boxes to supervision format
    # Gemini: [y0, x0, y1, x1] in 0-1000 range
    # Supervision: [x0, y0, x1, y1] in pixel coordinates
    
    img_array = np.array(image)
    height, width = img_array.shape[:2]
    
    xyxy_boxes = []
    labels = []
    
    for det in detections_data:
        y0, x0, y1, x1 = det['box_2d']
        
        # Convert from 1000-scale to pixel coordinates
        x0_px = int(x0 * width / 1000)
        y0_px = int(y0 * height / 1000)
        x1_px = int(x1 * width / 1000)
        y1_px = int(y1 * height / 1000)
        
        xyxy_boxes.append([x0_px, y0_px, x1_px, y1_px])
        labels.append(det['label'])
    
    # Create supervision Detections object
    detections = sv.Detections(
        xyxy=np.array(xyxy_boxes),
        class_id=np.arange(len(labels))
    )
    
    # Annotate with colored bounding boxes and labels
    box_annotator = sv.BoxAnnotator(
        thickness=3,
        color=sv.ColorPalette.from_hex(['#FF6B6B', '#4ECDC4'])
    )
    label_annotator = sv.LabelAnnotator(
        text_scale=0.6,
        text_thickness=2,
        text_color=sv.Color.WHITE
    )
    
    annotated_image = box_annotator.annotate(
        scene=img_array.copy(),
        detections=detections
    )
    annotated_image = label_annotator.annotate(
        scene=annotated_image,
        detections=detections,
        labels=labels
    )
    
    # Display side-by-side comparison
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
    
    ax1.imshow(image)
    ax1.set_title('Original Image', fontsize=14, fontweight='bold')
    ax1.axis('off')
    
    ax2.imshow(annotated_image)
    ax2.set_title(f'Detected: {", ".join(labels)}', fontsize=14, fontweight='bold')
    ax2.axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print("Detection complete!")
    print()
    print("Technical details:")
    print(f"- Gemini returns boxes in [y0, x0, y1, x1] format")
    print(f"- Coordinates scaled 0-1000 (normalized)")
    print(f"- Converted to pixel coords: {width}x{height}")
    print(f"- Visualized using supervision library")
    
except json.JSONDecodeError as e:
    print(f"Error parsing JSON: {e}")
    print(f"Response text: {response_text}")

## Image Segmentation

Now let's segment the cat to get a pixel-perfect mask

In [ ]:
# Request SVG segmentation from Gemini (faster than PNG mask)
segmentation_prompt = """Segment the cat in this image and return it as an SVG polygon.

Return a JSON object with:
- "label": "cat"
- "polygon": array of [x, y] coordinates in range [0, 1000] forming the outline

Example: {"label": "cat", "polygon": [[x1,y1], [x2,y2], [x3,y3], ...]}

Return ONLY the JSON object."""

print("Asking Gemini for SVG polygon segmentation...")

seg_response = client.models.generate_content(
    model=MODEL,
    contents=[segmentation_prompt, image]
)

print("Response:")
print(seg_response.text[:300] + "..." if len(seg_response.text) > 300 else seg_response.text)

In [ ]:
# Parse SVG polygon and visualize
from matplotlib.patches import Polygon as MPLPolygon

response_text = seg_response.text.strip()
if response_text.startswith('```'):
    lines = response_text.split('\n')
    response_text = '\n'.join(lines[1:-1])
    if response_text.startswith('json'):
        response_text = response_text[4:].strip()

seg_data = json.loads(response_text)

print(f"Label: {seg_data['label']}")
print(f"Polygon points: {len(seg_data['polygon'])}")

# Convert polygon from 1000-scale to pixel coordinates
img_array = np.array(image)
height, width = img_array.shape[:2]

polygon_coords = []
for x, y in seg_data['polygon']:
    px = int(x * width / 1000)
    py = int(y * height / 1000)
    polygon_coords.append([px, py])

polygon_coords = np.array(polygon_coords)

# Create mask from polygon
from matplotlib.path import Path
x, y = np.meshgrid(np.arange(width), np.arange(height))
points = np.c_[x.ravel(), y.ravel()]
path = Path(polygon_coords)
mask = path.contains_points(points).reshape(height, width)

# Create overlay
overlay = img_array.copy().astype(float)
color = np.array([255, 100, 100])
for c in range(3):
    overlay[:,:,c] = np.where(mask, overlay[:,:,c]*0.5 + color[c]*0.5, overlay[:,:,c])
overlay = overlay.astype(np.uint8)

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(image)
axes[0].set_title('Original', fontsize=14, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(image)
poly_patch = MPLPolygon(polygon_coords, fill=False, edgecolor='red', linewidth=2)
axes[1].add_patch(poly_patch)
axes[1].set_title('SVG Polygon Outline', fontsize=14, fontweight='bold')
axes[1].axis('off')

axes[2].imshow(overlay)
axes[2].set_title('Segmentation Overlay', fontsize=14, fontweight='bold')
axes[2].axis('off')

plt.tight_layout()
plt.show()

print(f"\nCoverage: {100*mask.sum()/mask.size:.1f}%")

## Mathematical Problem Solving from Images

Read a handwritten/printed math problem and solve it step-by-step

In [ ]:
# Load math problem image
math_image = Image.open('least-squares.jpg')

print("Math problem image loaded")
print(f"Size: {math_image.size}")

plt.figure(figsize=(12, 8))
plt.imshow(math_image)
plt.title('Original Math Problem', fontsize=14, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Ask Gemini to solve the math problem
math_prompt = """Look at this mathematical problem and solve it step by step.

Please:
1. First, identify what mathematical concepts/formulas are shown
2. Solve the problem completely, showing all steps
3. Explain each step clearly
4. Provide the final answer

Format your response as if you were writing it by hand on paper, with clear steps."""

print("Asking Gemini to solve the math problem...")
print()

math_response = client.models.generate_content(
    model=MODEL,
    contents=[math_prompt, math_image]
)

print("Gemini's Solution:")
print("="*80)
print(math_response.text)
print("="*80)

In [ ]:
# Generate handwritten-style solution image
from IPython.display import Markdown, display

# First show the markdown solution
print("Solution:")
print("="*80)
display(Markdown(math_response.text))
print("="*80)

# Now generate a handwritten-style image of the solution
IMAGE_MODEL = "models/gemini-3-pro-image-preview"

image_prompt = f"""Create a handwritten-style mathematical solution on paper.

The solution should show:
{math_response.text[:500]}

Style:
- Handwritten on white/cream paper
- Clear mathematical notation
- Step-by-step layout
- Professional but handwritten feel
- Include diagrams if helpful
- Blue/black ink style

Make it look like a student's neat homework solution."""

print("\nGenerating handwritten-style solution image...")

result = client.models.generate_content(
    model=IMAGE_MODEL,
    contents=image_prompt
)

# Display generated image
if result.candidates[0].content.parts:
    for part in result.candidates[0].content.parts:
        if hasattr(part, 'inline_data'):
            import base64
            from io import BytesIO
            
            image_data = base64.b64decode(part.inline_data.data)
            generated_img = Image.open(BytesIO(image_data))
            
            plt.figure(figsize=(14, 10))
            plt.imshow(generated_img)
            plt.title('AI-Generated Handwritten Solution', fontsize=14, fontweight='bold')
            plt.axis('off')
            plt.tight_layout()
            plt.show()
            
            print("\nGenerated handwritten-style solution image!")
else:
    print("No image generated")

## Visual Question Answering (VQA)

In [ ]:
print("\n" + "="*80)
print("Visual Question Answering")
print("="*80)

# Use sample images from URLs
try:
    image_url = "https://images.unsplash.com/photo-1506905925346-21bda4d32df4?w=800"
    image = load_image_from_url(image_url)

    # Show image
    plt.imshow(image)
    
    questions = [
        "What is the dominant color in this image?",
        "Describe the scenery",
        "What time of day does it appear to be?",
        "What mood does this image convey?"
    ]

    for q in questions:
        response = client.models.generate_content(
            model=MODEL,
            contents=[q, image]
        )
        print(f"Q: {q}")
        print(f"A: {response.text.strip()}\n")

except Exception as e:
    print(f"Note: Image loading requires internet. Error: {str(e)[:100]}")

## Chart and Graph Analysis


In [ ]:
print("\n" + "="*80)
print("Chart Analysis")
print("="*80)

# Create a complex chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Bar chart
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun']
sales = [45000, 52000, 48000, 61000, 58000, 72000]
ax1.bar(months, sales, color='steelblue')
ax1.set_title('Monthly Sales 2024', fontsize=14, fontweight='bold')
ax1.set_ylabel('Sales ($)')
ax1.grid(axis='y', alpha=0.3)

# Line chart
days = list(range(1, 31))
visitors = [100 + 50*np.sin(x/5) + np.random.randint(-10, 10) for x in days]
ax2.plot(days, visitors, marker='o', linewidth=2, markersize=4)
ax2.set_title('Daily Website Visitors', fontsize=14, fontweight='bold')
ax2.set_xlabel('Day of Month')
ax2.set_ylabel('Visitors')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/charts.png', dpi=150)


In [ ]:
plt.close()

chart_image = Image.open('/tmp/charts.png')

prompt = """Analyze these charts:
1. What trends do you see in the sales data?
2. Which month had the highest sales?
3. What pattern is visible in the website visitors chart?
4. Any notable insights?"""

response = client.models.generate_content(
    model=MODEL,
    contents=[prompt, chart_image])
print(response.text)

## Document OCR and Understanding

Gemini-3-Pro's **document understanding** goes beyond simple OCR. It can extract, interpret, and reason about text in complex layouts including receipts, forms, and multi-column documents. This is powered by Gemini-3-Pro's exceptional performance on document understanding benchmarks.


In [ ]:
print("\n" + "="*80)
print("Document OCR + Understanding")
print("="*80)

# Create a sample receipt
img = Image.new('RGB', (600, 800), color='white')
draw = ImageDraw.Draw(img)

receipt_lines = [
    "ACME STORE",
    "123 Main Street",
    "Phone: (555) 123-4567",
    "",
    "Date: 2024-12-01",
    "Receipt #: 45678",
    "-" * 40,
    "Coffee Beans (2kg)      $24.99",
    "Milk (1L)                $3.49",
    "Bread                    $2.99",
    "Fresh Vegetables        $12.50",
    "-" * 40,
    "Subtotal:              $43.97",
    "Tax (8%):               $3.52",
    "TOTAL:                 $47.49",
    "",
    "Payment: VISA ****1234",
    "Thank you for shopping!"
]

y = 50
for line in receipt_lines:
    draw.text((50, y), line, fill='black')
    y += 35

img.save('/tmp/receipt.png')
receipt_img = Image.open('/tmp/receipt.png')
plt.imshow(receipt_img)
plt.axis('off')



In [ ]:
prompt = """Extract information from this receipt:
1. Store name and address
2. Date and receipt number
3. List of items purchased with prices
4. Total amount
5. Payment method

Format as structured JSON."""

response = client.models.generate_content(
    model=MODEL,
    contents=[prompt, receipt_img])
print(response.text)

## Video Capabilities

In [ ]:
video_path = '14801276_2160_3840_30fps.mp4'
from IPython.display import Video

# Display video with constrained width (max 600px)
Video(video_path, embed=True, height=400)

In [ ]:
print("\n" + "="*80)
print("Video Understanding")
print("="*80)

import time


print(f"Uploading video: {video_path}")

video_file = client.files.upload(file=video_path)
print(f"Video uploaded: {video_file.name}")

# Wait for processing
print("Processing video...")
while video_file.state == 'PROCESSING':
    time.sleep(2)
    video_file = client.files.get(name=video_file.name)
print(".", end="", flush=True)
print(f"\nVideo ready! State: {video_file.state}")

# Analyze video
prompt = 'Describe what happens in this video. What do you see?'
print(f"\nQuery: {prompt}\n")

response = client.models.generate_content(
    model=MODEL,
    contents=[prompt, video_file]
)
print(f"Response:\n{response.text}")

## Video Frame Analysis


In [ ]:
print("\n" + "="*80)
print("Action Recognition in Videos")
print("="*80)

# Analyze specific actions in the video
action_queries = [
    "List all distinct visual elements or objects in chronological order as they appear",
]

for query in action_queries:
    print(f"\nQuery: {query}")

    response = client.models.generate_content(
        model=MODEL,
        contents=[query, video_file]
    )
    print(f"Answer: {response.text}\n")
    print("-" * 80)

# Audio Processing



In [ ]:
from IPython.display import Audio

audio_path = "sample-audio.mp3"
Audio(audio_path, embed=True)

## Speech Transcription


In [ ]:
print("\n" + "="*80)
print("Audio Transcription")
print("="*80)

import time


audio_file = client.files.upload(file=audio_path)
print(f"Audio uploaded: {audio_file.name}")

# Wait for processing
print("Processing audio...")
while audio_file.state == 'PROCESSING':
    time.sleep(1)
    audio_file = client.files.get(name=audio_file.name)
print(".", end="", flush=True)
print(f"\nAudio ready! State: {audio_file.state}")

# Transcribe
print("\nTranscription:\n")
response = client.models.generate_content(
    model=MODEL,
    contents=[
        'Transcribe this audio exactly as spoken. Include any speech, sounds, or notable audio features.',
        audio_file
    ]
)
print(response.text)

## Image Generation

Gemini can also **generate images** using natural language prompts. 

In [ ]:
print("\n" + "="*80)
print("Image Generation: Research Infographic")
print("="*80)


prompt = """Create a clean, modern infographic illustrating the concept of 
"Transfer Learning in Deep Neural Networks".

The infographic should show:
- A pre-trained model (represented as a neural network)
- An arrow showing transfer to a new task
- The fine-tuning process
- Use a professional color scheme (blues and purples)
- Include minimal text labels
- Make it suitable for a research presentation

Style: Clean, minimal, professional, technical illustration"""

result = client.models.generate_content(
    model=IMAGE_MODEL,
    contents=prompt
)

print(f"Generated {len(result.candidates[0].content.parts)} image(s)")


In [ ]:
# Display the generated image
import base64
from io import BytesIO

if result.candidates and result.candidates[0].content.parts:
    for part in result.candidates[0].content.parts:
        if hasattr(part, 'inline_data'):
            # Decode base64 image data
            image_data = base64.b64decode(part.inline_data.data)
            generated_img = Image.open(BytesIO(image_data))
            
            # Display with matplotlib for better control
            plt.figure(figsize=(12, 8))
            plt.imshow(generated_img)
            plt.title('Generated Image', fontsize=14, fontweight='bold')
            plt.axis('off')
            plt.tight_layout()
            plt.show()
            
            print(f"Generated image size: {generated_img.size}")
            break
else:
    print("No image was generated in the response")
    print(f"Response: {result}")

In [ ]:
print("\n" + "="*80)
print("SVG Logo Generation")
print("="*80)

# Generate SVG logo using text generation (not image generation)
prompt = """Create an elegant SVG logo for an AI research lab called "Neural Insights".

Requirements:
- Return ONLY valid SVG code (starting with <svg> tag)
- Use a brain or neural network motif
- Incorporate the text "Neural Insights"
- Use a modern, minimalist design
- Color scheme: gradient from #667eea to #764ba2
- Size: 400x200 viewBox
- Professional and clean

Return ONLY the SVG code, no explanation."""

result = client.models.generate_content(
    model=MODEL,
    contents=prompt
)

svg_code = result.text.strip()
print(f"Generated SVG ({len(svg_code)} characters)")


In [ ]:
# Display the SVG logo
from IPython.display import SVG, display

# Clean up the SVG code
if '```' in svg_code:
    svg_code = svg_code.split('```')[1]
    if svg_code.startswith('svg'):
        svg_code = svg_code[3:].strip()
svg_code = svg_code.strip()

# Extract just the SVG part if there's extra text
if '<svg' in svg_code:
    start = svg_code.index('<svg')
    end = svg_code.rindex('</svg>') + 6
    svg_code = svg_code[start:end]

print("\nRendered SVG Logo:")
display(SVG(svg_code))


## Spatial Reasoning and Pointing



In [ ]:
print("\n" + "="*80)
print("Spatial Reasoning: Object Relationships")
print("="*80)

# Create an image with objects in different spatial relationships
img = Image.new('RGB', (800, 600), color='white')
draw = ImageDraw.Draw(img)

# Draw a table (brown rectangle)
draw.rectangle([200, 400, 600, 550], fill='#8B4513', outline='black', width=3)

# Draw a cup ON the table (blue circle)
draw.ellipse([350, 340, 450, 420], fill='lightblue', outline='blue', width=3)

# Draw a book UNDER the cup (green rectangle)
draw.rectangle([320, 380, 420, 430], fill='lightgreen', outline='green', width=2)

# Draw a lamp BESIDE the cup (yellow with black base)
draw.ellipse([520, 300, 560, 340], fill='yellow', outline='orange', width=2)
draw.rectangle([535, 340, 545, 410], fill='black')

# Draw a picture frame ABOVE the table (red rectangle on wall)
draw.rectangle([300, 150, 500, 300], fill='lightcoral', outline='red', width=3)

img.save('/tmp/spatial_scene.png')
plt.figure(figsize=(10, 7))
plt.imshow(img)
plt.axis('off')
plt.title('Scene with Spatial Relationships')
plt.tight_layout()
plt.show()

prompt = """Analyze the spatial relationships in this image:
1. List all objects you can identify
2. For each object, describe its position relative to other objects (above, below, on, beside, etc.)
3. Which object is at the highest position?
4. Which objects are in direct contact with each other?
5. If I wanted to pick up the cup, what other object would I need to move first?"""

spatial_image = Image.open('/tmp/spatial_scene.png')
result = client.models.generate_content(
    model=MODEL,
    contents=[prompt, spatial_image]
)
print(f"\nSpatial Analysis Result:\n{result.text}")


# Document Intelligence (PDF Analysis)


## PDF Document Analysis


In [ ]:
print("\n" + "="*80)
print("PDF Document Understanding")
print("="*80)

import time

# Upload PDF (using available PDF file)
pdf_path = 'batra_nilmtk.pdf'
print(f"Uploading PDF: {pdf_path}")

pdf_file = client.files.upload(file=pdf_path)
print(f"PDF uploaded: {pdf_file.name}")

# Wait for processing
print("Processing PDF...")
while pdf_file.state == 'PROCESSING':
    time.sleep(2)
    pdf_file = client.files.get(name=pdf_file.name)
print(".", end="", flush=True)
print(f"\nPDF ready! State: {pdf_file.state}")

# Analyze document
prompts = [
    "What type of document is this? Provide a brief overview.",
    "What are the main sections or topics covered?",
    "Summarize the key points in 3-4 sentences."
]

for prompt in prompts:
    print(f"\nQuery: {prompt}")

    response = client.models.generate_content(
        model=MODEL,
        contents=[prompt, pdf_file]
    )
print(f"Answer: {response.text}\n")
print("-" * 80)

## Multi-Page PDF Extraction


In [ ]:
print("\n" + "="*80)
print("Multi-Page PDF Data Extraction")
print("="*80)

# Extract structured information from the PDF
extraction_prompt = """Extract the following information from this PDF document:
1. Title/heading
2. Authors (if any)
3. Main topics or sections
4. Any key findings or conclusions
5. Number of pages (estimate)

Format your response as a structured summary."""

print("Extracting structured data from PDF...")
print()

response = client.models.generate_content(
    model=MODEL,
    contents=[extraction_prompt, pdf_file]
)
print(response.text)
print("\n" + "="*80)
print("PDF analysis demonstrates:")
print("- Multi-page document understanding")
print("- Structure extraction (headings, sections)")
print("- Content summarization")
print("- Data extraction from mixed content (text, tables, figures)")

## Structured JSON Output


In [ ]:
print("\n" + "="*80)
print("Structured JSON Output")
print("="*80)

text = """Sarah Johnson, 34, is a Senior Data Scientist at TechCorp in San Francisco. 
She specializes in machine learning and has 8 years of experience. Her skills include 
Python, TensorFlow, and SQL. Contact: sarah.j@techcorp.com, (555) 987-6543."""

schema = {
    "name": " ",
    "age": 0,
    "title": "",
    "company": "",
    "location": "",
    "experience_years": 0,
    "skills": [],
    "contact": {
        "email": "",
        "phone": ""
    }
}

prompt = f"""Extract information and return valid JSON matching this schema:
{json.dumps(schema, indent=2)}

Text: {text}

Return ONLY the JSON object, no markdown formatting:"""

response = client.models.generate_content(
    model=MODEL,
    contents=prompt)
result = response.text.strip()

# Clean up markdown if present
if result.startswith('```'):
    result = '\n'.join(result.split('\n')[1:-1])
    if result.startswith('json'):
        result = result[4:]

try:
    parsed = json.loads(result.strip())
    print("Extracted JSON:")
    print(json.dumps(parsed, indent=2))
except json.JSONDecodeError as e:
    print(f"Raw output:\n{result}")
    print(f"\nJSON parse error: {e}")   

## Search Grounding

In [ ]:
print("\n" + "="*80)
print("Search Grounding (Google Search Integration)")
print("="*80)

from google.genai import types

# Enable Google Search grounding
google_search_tool = types.Tool(google_search={})

# Ask time-sensitive questions
queries = [
    "What are the latest developments in AI announced this month?",
    "Who won the latest Nobel Prize in Physics?",
]

print("Using Google Search grounding for real-time information:\n")

for i, query in enumerate(queries, 1):
    print(f"{i}. Query: {query}")

    try:
        response = client.models.generate_content(
            model=MODEL,
            contents=query,
            config=types.GenerateContentConfig(
                tools=[google_search_tool]
            )
        )
        print(f"Answer: {response.text}")

        # Try to access grounding metadata if available
        if hasattr(response, 'grounding_metadata') and response.grounding_metadata:
            print(f"Response is grounded with search results")
            if hasattr(response.grounding_metadata, 'web_search_queries'):
                print(f"  Search queries used: {response.grounding_metadata.web_search_queries}")
        print()

    except Exception as e:
        print(f"Note: {str(e)[:100]}")
        print("Search grounding may not be available for all models/regions")
        print()
        break



print("\n" + "="*80)
print("Code Generation")
print("="*80)

code_tasks = [
    {
        "language": "Python",
        "task": """Create a decorator that measures function execution time
        and logs it with the function name."""
    },
    {
        "language": "JavaScript",
        "task": """Create an async function that fetches data from multiple
        URLs in parallel and returns combined results."""
    },
    {
        "language": "SQL",
        "task": """Write a query to find the top 5 customers by total
        purchase amount in the last 30 days."""
    }
]

for task in code_tasks:
    prompt = f"""Write {task['language']} code for this task:

{task['task']}

Include:
- Clean, production-ready code
- Type hints/comments where appropriate
- Error handling
- A brief explanation
"""

    response = client.models.generate_content(
        model="gemini-2.0-flash-thinking-exp-1219",
        contents=prompt
    )
    print(f"\n{task['language']} Task: {task['task'][:50]}...")
    print("-" * 80)
    print(response.text)
    print()

print("\n" + "="*80)
print("Mathematical Problem Solving")
print("="*80)

math_problems = [
    {
        "type": "Calculus",
        "problem": "Find the derivative of f(x) = (3x² + 2x - 1) * e^x"
    },
    {
        "type": "Linear Algebra",
        "problem": """Find the eigenvalues of the matrix:
        [[2, 1],
         [1, 2]]"""
    },
    {
        "type": "Statistics",
        "problem": """Given data: [12, 15, 18, 22, 25, 30, 35]
        Calculate: mean, median, variance, and standard deviation"""
    },
    {
        "type": "Optimization",
        "problem": """A rectangular garden has perimeter of 60m.
        What dimensions maximize the area?"""
    }
]

for prob in math_problems:
    prompt = f"""Solve this {prob['type']} problem step by step:

{prob['problem']}

Show all work and explain each step."""

    response = client.models.generate_content(
        model="gemini-2.0-flash-thinking-exp-1219",
        contents=prompt
    )
    print(f"\n{prob['type']}: {prob['problem'][:50]}...")
    print("-" * 80)
    print(response.text)
    print()

# Conclusion

Gemini-3-Pro represents a **breakthrough in multimodal AI**, delivering state-of-the-art performance across vision, video, audio, and document understanding. Through this comprehensive exploration, we've demonstrated:



## Resources

-  [Gemini-3-Pro Official Announcement](https://blog.google/technology/developers/gemini-3-pro-vision/)
-  [Gemini API Documentation](https://ai.google.dev/gemini-api/docs)
-  [More Gemini-3 Examples](https://blog.google/products/gemini/gemini-3-examples-demos/)

## Attribution

> **Note**: Several examples and concepts in this notebook are inspired by demonstrations from Google's [Gemini-3-Pro announcement](https://blog.google/technology/developers/gemini-3-pro-vision/), including derendering capabilities, high-FPS video analysis, and spatial reasoning tasks. All examples are implemented using the official Gemini API.

---
